# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [16]:
import scipy.io
import numpy as np

# ==================== 1️⃣ 读取 .mat 文件 ====================
mat_data = scipy.io.loadmat(
    '/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_3.mat'
)

data = mat_data['Unit03_1_5_select_3']

print("Original data shape:", data.shape)  # (N, 14)

# ==================== 2️⃣ 设置抽样参数 ====================
n = 5000        # ⭐你只需要改这里
seed = 42      # 可选：保证可复现

np.random.seed(seed)

num_samples = data.shape[0]
assert n <= num_samples, "n cannot be larger than total samples!"

# ==================== 3️⃣ 随机抽取 n 个样本（行） ====================
indices = np.random.choice(num_samples, size=n, replace=False)
data = data[indices, :]

print("Sampled data shape:", data.shape)

# ==================== 4️⃣ 查看抽样结果 ====================
print(data[:5])   # 前 5 行看看


Original data shape: (14000, 14)
Sampled data shape: (5000, 14)
[[ 5.43047428e+00  1.05059376e+01  2.31127650e-01 -1.40501881e+01
   2.50210800e+02  3.08711639e+02  5.54433655e+02  8.17137878e+02
   8.00413025e+02  8.91505981e+02  7.48925903e+02  7.47335205e+02
   7.47687561e+02  1.22015419e+01]
 [ 5.43321800e+00  1.05319557e+01  2.27354318e-01 -1.53432178e+01
   2.44723053e+02  3.09451508e+02  5.57394836e+02  8.27757263e+02
   8.06735474e+02  8.95841003e+02  7.50904175e+02  7.50025269e+02
   7.50035461e+02  1.33322163e+01]
 [ 6.86902332e+00  9.63033295e+00  2.11065754e-01 -1.29801950e+01
   2.62162231e+02  3.07413025e+02  5.59327209e+02  8.18008972e+02
   7.97732300e+02  8.90310486e+02  7.49348816e+02  7.47007019e+02
   7.47838074e+02  6.77845287e+00]
 [ 5.36784887e+00  1.06744299e+01  2.37428218e-01 -1.19700117e+01
   2.54997070e+02  3.07855316e+02  5.53981567e+02  8.24121216e+02
   8.05774719e+02  8.93992188e+02  7.48753174e+02  7.47304749e+02
   7.47581665e+02  9.15509033e+00]
 [ 6

## 特征和标签的分离

In [17]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (5000, 13)
Shape of Labels (y): (5000, 1)


## 三集划分

In [18]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (3499, 13) (3499, 1)
Shape of Validation Set (X_val, y_val): (750, 13) (750, 1)
Shape of Test Set (X_test, y_test): (751, 13) (751, 1)


## 归一化

In [19]:
import numpy as np

# ======================= 1️⃣ X：用 train 的均值和方差 =======================
mu = X_train.mean(axis=0, keepdims=True)     # (1, 13)
std = X_train.std(axis=0, keepdims=True)    # (1, 13)
std[std == 0] = 1e-8

X_train = (X_train - mu) / std
X_val   = (X_val   - mu) / std
X_test  = (X_test  - mu) / std

print("X normalized shapes:")
print(X_train.shape, X_val.shape, X_test.shape)


# ======================= 2️⃣ y：同样只用 train =======================
y_mu = y_train.mean(axis=0, keepdims=True)     # (1,1)
y_std = y_train.std(axis=0, keepdims=True)    # (1,1)
y_std[y_std == 0] = 1e-8

y_train = (y_train - y_mu) / y_std
y_val   = (y_val   - y_mu) / y_std
y_test  = (y_test  - y_mu) / y_std

print("y normalized shapes:")
print(y_train.shape, y_val.shape, y_test.shape)


# ======================= 3️⃣ 反归一化函数 =======================
def inverse_y(y_norm, y_mu, y_std):
    """
    y_norm: normalized prediction, shape [N,1] or [N]
    y_mu, y_std: from training set
    """
    return y_norm * y_std + y_mu


X normalized shapes:
(3499, 13) (750, 13) (751, 13)
y normalized shapes:
(3499, 1) (750, 1) (751, 1)


# 图结构搭建

In [20]:
import numpy as np
import torch
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 输入：
#   X : numpy.ndarray or torch.Tensor
#       shape = [num_samples, num_features]
# 输出：
#   adj : torch.Tensor
#       shape = [num_features, num_features]
# =========================================================

def build_feature_graph_from_X(X, threshold=0.4, device="cpu"):
    """
    Automatically build a feature (column-wise) graph from X.

    Parameters
    ----------
    X : np.ndarray or torch.Tensor
        Shape [num_samples, num_features]
    threshold : float
        Cosine similarity threshold for edge creation
    device : str or torch.device
        cpu / cuda

    Returns
    -------
    adj : torch.Tensor
        Adjacency matrix of shape [num_features, num_features]
    """

    # ---------- 1️⃣ 统一成 numpy ----------
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X

    # ---------- 2️⃣ 自动读取形状 ----------
    num_samples, num_features = X_np.shape
    print(f"[INFO] X shape: samples={num_samples}, features={num_features}")

    # ---------- 3️⃣ 特征（列）作为节点 ----------
    # 每一列是一个节点向量（跨样本）
    X_feature = X_np.T                         # [F, M]

    # ---------- 4️⃣ 特征间相似度 ----------
    sim_matrix = cosine_similarity(X_feature) # [F, F]

    # ---------- 5️⃣ 构建 NetworkX 图 ----------
    G = nx.Graph()
    G.add_nodes_from(range(num_features))

    for i in range(num_features):
        for j in range(i + 1, num_features):
            if sim_matrix[i, j] >= threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    print(f"[INFO] Graph built: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    # ---------- 6️⃣ Graph → 邻接矩阵 ----------
    adj_np = nx.to_numpy_array(G, weight="weight")  # [F, F]

    # ---------- 7️⃣ 转成 torch.Tensor ----------
    adj = torch.tensor(adj_np, dtype=torch.float32, device=device)

    # ---------- 8️⃣ 简单健壮性检查 ----------
    isolated = (adj.sum(dim=1) == 0).sum().item()
    if isolated > 0:
        print(f"[WARN] {isolated} isolated feature nodes detected "
              f"(consider lowering threshold or using KNN graph)")

    print(f"[INFO] adj shape: {adj.shape}")
    return adj


device = "cuda" if torch.cuda.is_available() else "cpu"

adj = build_feature_graph_from_X(X_train, threshold=0.3, device=device)

print(adj)


[INFO] X shape: samples=3499, features=13
[INFO] Graph built: nodes=13, edges=26
[INFO] adj shape: torch.Size([13, 13])
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.9219, 0.0000, 0.5171, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.9795, 0.0000, 0.0000, 0.6433, 0.0000, 0.5955, 0.5272,
         0.3124, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9795, 0.0000, 0.0000, 0.0000, 0.6623, 0.0000, 0.6200, 0.5829,
         0.3576, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4169, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.9219, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5484, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6433, 0.6623, 0.0000, 0.0000, 0.0000, 0.3391, 0.6167, 0.5103,
         0.4695, 0.0000, 0.0000, 0.0000],
        [0.5171, 0.0000, 0.0000, 0.0000, 0.5484, 0.3391, 0.0000, 0.5402, 0.4343,
         0.7179, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.

# 模型

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BSpline1D(nn.Module):
    """
    Simple learnable 1D B-spline (uniform knots) for KAN edges.
    Input:  [B, in_dim]
    Output: [B, out_dim]
    """
    def __init__(self, in_dim, out_dim, num_knots=8):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_knots = num_knots

        # 控制点系数（每个输入维到每个输出维）
        self.coeff = nn.Parameter(torch.randn(in_dim, out_dim, num_knots) * 0.01)

        # 均匀结点（固定）
        self.register_buffer("knots", torch.linspace(-3, 3, num_knots))

    def forward(self, x):
        # x: [B, in_dim]
        # 计算每个 knot 的基函数权重（高斯核近似 B-spline，稳定可导）
        # phi: [B, in_dim, num_knots]
        x_exp = x.unsqueeze(-1)                         # [B, in_dim, 1]
        phi = torch.exp(-((x_exp - self.knots)**2))    # RBF 近似 spline basis
        phi = phi / (phi.sum(dim=-1, keepdim=True) + 1e-8)

        # 加权求和到 out_dim
        # out[b, o] = sum_i sum_k phi[b,i,k] * coeff[i,o,k]
        out = torch.einsum('bik,iok->bo', phi, self.coeff)
        return out


class KANLayer(nn.Module):
    """
    One KAN layer: y = Wb * SiLU(x) + Ws * Spline(x)
    """
    def __init__(self, in_dim, out_dim, num_knots=8):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=True)   # Wb * SiLU(x)
        self.spline = BSpline1D(in_dim, out_dim, num_knots=num_knots)  # Ws * Spline(x)

    def forward(self, x):
        return self.lin(F.silu(x)) + self.spline(x)


class KANRegressor(nn.Module):
    """
    Prediction head used in AKGNN (KAN).
    Input:  g [B, D]   (graph-aggregated features)
    Output: y [B, 1]
    """
    def __init__(self, in_dim, hidden=64, num_knots=8):
        super().__init__()
        self.kan1 = KANLayer(in_dim, hidden, num_knots=num_knots)
        self.kan2 = KANLayer(hidden, hidden, num_knots=num_knots)
        self.out  = nn.Linear(hidden, 1)

    def forward(self, g):
        h = self.kan1(g)
        h = self.kan2(h)
        y = self.out(h)
        return y



In [30]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.metrics import r2_score

# ======================= 0️⃣ numpy → torch =======================

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# ======================= 1️⃣ Dataset / Loader =======================

batch_size = 64

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=batch_size,
    shuffle=False
)

# ======================= 2️⃣ Model =======================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

D = X_train.shape[1]   # 13

model = KANRegressor(in_dim=D, hidden=64, num_knots=8).to(device)

# ======================= 3️⃣ Optimizer / Loss =======================

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# ======================= 4️⃣ Train / Val Loop =======================

num_epochs = 100

best_val_rmse = float("inf")
SAVE_PATH = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/transfer_learning_v1/result/model_save/best_kan_model.pt"

for epoch in range(1, num_epochs + 1):

    # -------- Train --------
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)

        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    # -------- Validation --------
    model.eval()
    val_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += criterion(pred, yb).item() * xb.size(0)

            y_true.append(yb.cpu().numpy())
            y_pred.append(pred.cpu().numpy())

    val_loss /= len(val_loader.dataset)

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    val_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    val_r2 = r2_score(y_true, y_pred)

    # ======================= ⭐ 保存最优模型 =======================
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"✅ Best model updated at epoch {epoch}, Val RMSE = {val_rmse:.4f}")

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"[Epoch {epoch:03d}] "
            f"Train MSE: {train_loss:.4f} | "
            f"Val RMSE: {val_rmse:.4f} | "
            f"Val R2: {val_r2:.4f}"
        )

print(f"\n🎯 Training finished. Best Val RMSE: {best_val_rmse:.4f}")
print(f"Model saved to: {SAVE_PATH}")



/tmp/ipykernel_47857/801500643.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_t = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipykernel_47857/801500643.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_t = torch.tensor(y_train, dtype=torch.float32)
/tmp/ipykernel_47857/801500643.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val_t = torch.tensor(X_val, dtype=torch.float32)
/tmp/ipykernel_47857/801500643.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().de

✅ Best model updated at epoch 1, Val RMSE = 0.7782
[Epoch 001] Train MSE: 0.6322 | Val RMSE: 0.7782 | Val R2: 0.4054
✅ Best model updated at epoch 2, Val RMSE = 0.7257
✅ Best model updated at epoch 3, Val RMSE = 0.6962
✅ Best model updated at epoch 4, Val RMSE = 0.6635
✅ Best model updated at epoch 5, Val RMSE = 0.6490
✅ Best model updated at epoch 6, Val RMSE = 0.6450
✅ Best model updated at epoch 7, Val RMSE = 0.6117
✅ Best model updated at epoch 8, Val RMSE = 0.5740
✅ Best model updated at epoch 9, Val RMSE = 0.5681
✅ Best model updated at epoch 10, Val RMSE = 0.5634
[Epoch 010] Train MSE: 0.3059 | Val RMSE: 0.5634 | Val R2: 0.6884
✅ Best model updated at epoch 11, Val RMSE = 0.5552
✅ Best model updated at epoch 12, Val RMSE = 0.5410
✅ Best model updated at epoch 14, Val RMSE = 0.5397
✅ Best model updated at epoch 15, Val RMSE = 0.5237
✅ Best model updated at epoch 16, Val RMSE = 0.5175
✅ Best model updated at epoch 17, Val RMSE = 0.5036
✅ Best model updated at epoch 19, Val RMSE = 

# 测试

In [35]:
from sklearn.metrics import r2_score
import numpy as np
import torch

# ======================= 0️⃣ numpy → torch =======================

X_test_t = torch.tensor(X_train, dtype=torch.float32)
y_test_t = torch.tensor(y_train, dtype=torch.float32)


@torch.no_grad()
def evaluate_r2(model, X_data, y_data, batch_size):
    model.eval()

    y_true_list = []
    y_pred_list = []

    num_samples = X_data.shape[0]

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb)  # ⭐ only X

        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)

    return r2_score(y_true, y_pred)

r2 = evaluate_r2(model, X_test_t, y_test_t, batch_size)
print(f"✅ Test R²: {r2:.4f}")


✅ Test R²: 0.9136


/tmp/ipykernel_47857/3137837325.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_t = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipykernel_47857/3137837325.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test_t = torch.tensor(y_train, dtype=torch.float32)
